# Low-RMSD train similarity regression

Compare how well **PLM Wasserstein distances** and **concat-CDR sequence identity** explain test-set CDRH3 RMSD (`H_cdr3`), when test samples are matched only against the **1,536 well-learned training antibodies** in `train_meta_low_rmsd_clusters.csv`.

- Wasserstein distance: **lower = more similar** (expect positive slope vs RMSD)
- Sequence identity: **higher = more similar** (expect negative slope vs RMSD)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#LOW_RMSD_TRAIN_PATH = '/opig-shared/users/lina4783/structures_final/train_meta_low_rmsd_clusters.csv'
LOW_RMSD_TRAIN_PATH = '/opig-shared/users/lina4783/structures_final/train_meta.csv'
TRAIN_META_PATH = '/opig-shared/users/lina4783/structures_final/train_meta.csv'
TEST_META_PATH = '/opig-shared/users/lina4783/structures_final/test_meta.csv'
#RMSD_SUMMARY_PATH = '/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_sample_imgt/struc_pred_metrics_test_summary.csv'
RMSD_SUMMARY_PATH = '/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_ckpt_5139_imgt/struc_pred_metrics_test_summary.csv'
WASSERSTEIN_MATRIX_PATH = '/opig-shared/users/lina4783/abb4_experiments/plm/wasserstein_out/vhvl_wasserstein_cdrs/similarity_matrix.csv'
IDENTITY_MATRIX_PATH = '/opig-shared/users/lina4783/cdrh3_similarity_out/train_test_concat_CDR/similarity_matrix.csv'
FEATURES_OUT_PATH = '/opig-shared/users/lina4783/abb4_experiments/evaluation/low_rmsd_train_features_newrun.csv'

TARGET = 'H_cdr3'
KNN_KS = (3, 5, 10)
WASSERSTEIN_RADII = (1100, 1300, 1500)
IDENTITY_THRESHOLDS = (0.6, 0.7, 0.8)
COVARIATES = ['cdrh3_len', 'concat_cdr_len', 'H_cdr3_var']

FEATURE_COLS_WASSERSTEIN = [
    'low_rmsd_best_wasserstein_distance',
    'low_rmsd_wasserstein_mean_knn_3',
    'low_rmsd_wasserstein_mean_knn_5',
    'low_rmsd_wasserstein_mean_knn_10',
    'low_rmsd_wasserstein_best_same_cdrh3_len',
    'low_rmsd_wasserstein_best_same_concat_cdr_len',
    'low_rmsd_n_train_wasserstein_lte_1100',
    'low_rmsd_n_train_wasserstein_lte_1300',
    'low_rmsd_n_train_wasserstein_lte_1500',
    'low_rmsd_wasserstein_match_cluster_size',
]

FEATURE_COLS_IDENTITY = [
    'low_rmsd_best_identity_concat_cdr',
    'low_rmsd_identity_mean_topk_3',
    'low_rmsd_identity_mean_topk_5',
    'low_rmsd_identity_mean_topk_10',
    'low_rmsd_identity_best_same_cdrh3_len',
    'low_rmsd_identity_best_same_concat_cdr_len',
    'low_rmsd_n_train_identity_gt_0_6',
    'low_rmsd_n_train_identity_gt_0_7',
    'low_rmsd_n_train_identity_gt_0_8',
    'low_rmsd_identity_match_cluster_size',
]

In [ ]:
def univariate_linear_regression(x, y):
    """Fit y = intercept + slope * x; return R2, adjusted R2, slope, intercept, p_value."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(y)

    slope, intercept = np.polyfit(x, y, 1)
    y_pred = slope * x + intercept
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot else 0.0
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - 2) if n > 2 else np.nan

    df = n - 2
    x_centered = x - x.mean()
    denom = np.sum(x_centered ** 2)
    se_slope = np.sqrt(ss_res / df / denom) if df > 0 and denom else np.nan
    t_stat = slope / se_slope if se_slope else np.nan
    try:
        from scipy.stats import t as t_dist
        p_value = 2.0 * t_dist.sf(np.abs(t_stat), df)
    except ImportError:
        p_value = np.nan

    return {
        'n': n,
        'slope': slope,
        'intercept': intercept,
        'r2': r2,
        'adj_r2': adj_r2,
        'r2_percent': 100.0 * r2,
        'p_value': p_value,
    }


def multivariate_linear_regression(X, y):
    """Fit y = X @ beta with intercept; return R2, adjusted R2, coefficients."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.all(np.isfinite(X), axis=1) & np.isfinite(y)
    X, y = X[mask], y[mask]
    n, p = X.shape

    design = np.column_stack([np.ones(n), X])
    beta, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
    y_pred = design @ beta
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot else 0.0
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1) if n > p + 1 else np.nan

    return {
        'n': n,
        'p': p,
        'r2': r2,
        'adj_r2': adj_r2,
        'r2_percent': 100.0 * r2,
        'intercept': beta[0],
        'coefficients': beta[1:],
    }


def masked_row_min(values, match_mask):
    masked = np.where(match_mask, values, np.inf)
    mins = masked.min(axis=1)
    mins[~match_mask.any(axis=1)] = np.nan
    return mins


def masked_row_max(values, match_mask):
    masked = np.where(match_mask, values, -np.inf)
    maxs = masked.max(axis=1)
    maxs[~match_mask.any(axis=1)] = np.nan
    return maxs


def compute_distance_features(
    dist_mat,
    train_ids,
    train_cdrh3_len,
    train_concat_len,
    test_cdrh3_len,
    test_concat_len,
    id_to_cluster,
    cluster_sizes,
    knn=KNN_KS,
    radii=WASSERSTEIN_RADII,
):
    train_ids = np.asarray(train_ids)
    best_j = dist_mat.argmin(axis=1)
    best_train_ids = train_ids[best_j]

    out = {
        'low_rmsd_best_train_id_wasserstein': best_train_ids,
        'low_rmsd_best_wasserstein_distance': dist_mat[np.arange(dist_mat.shape[0]), best_j],
    }
    for k in knn:
        out[f'low_rmsd_wasserstein_mean_knn_{k}'] = np.partition(dist_mat, k - 1, axis=1)[:, :k].mean(axis=1)

    cdrh3_match = test_cdrh3_len[:, None] == train_cdrh3_len[None, :]
    concat_match = test_concat_len[:, None] == train_concat_len[None, :]
    out['low_rmsd_wasserstein_best_same_cdrh3_len'] = masked_row_min(dist_mat, cdrh3_match)
    out['low_rmsd_wasserstein_best_same_concat_cdr_len'] = masked_row_min(dist_mat, concat_match)

    for r in radii:
        out[f'low_rmsd_n_train_wasserstein_lte_{r}'] = (dist_mat <= r).sum(axis=1)

    match_cluster = pd.Series(best_train_ids).map(id_to_cluster).to_numpy()
    out['low_rmsd_wasserstein_match_cluster'] = match_cluster
    out['low_rmsd_wasserstein_match_cluster_size'] = pd.Series(match_cluster).map(cluster_sizes).to_numpy()
    return out


def compute_similarity_features(
    sim_mat,
    train_ids,
    train_cdrh3_len,
    train_concat_len,
    test_cdrh3_len,
    test_concat_len,
    id_to_cluster,
    cluster_sizes,
    topk=KNN_KS,
    thresholds=IDENTITY_THRESHOLDS,
):
    train_ids = np.asarray(train_ids)
    best_j = sim_mat.argmax(axis=1)
    best_train_ids = train_ids[best_j]

    out = {
        'low_rmsd_best_train_id_identity': best_train_ids,
        'low_rmsd_best_identity_concat_cdr': sim_mat[np.arange(sim_mat.shape[0]), best_j],
    }
    for k in topk:
        topk_vals = np.partition(sim_mat, sim_mat.shape[1] - k, axis=1)[:, -k:]
        out[f'low_rmsd_identity_mean_topk_{k}'] = topk_vals.mean(axis=1)

    cdrh3_match = test_cdrh3_len[:, None] == train_cdrh3_len[None, :]
    concat_match = test_concat_len[:, None] == train_concat_len[None, :]
    out['low_rmsd_identity_best_same_cdrh3_len'] = masked_row_max(sim_mat, cdrh3_match)
    out['low_rmsd_identity_best_same_concat_cdr_len'] = masked_row_max(sim_mat, concat_match)

    for t in thresholds:
        key = str(t).replace('.', '_')
        out[f'low_rmsd_n_train_identity_gt_{key}'] = (sim_mat >= t).sum(axis=1)

    match_cluster = pd.Series(best_train_ids).map(id_to_cluster).to_numpy()
    out['low_rmsd_identity_match_cluster'] = match_cluster
    out['low_rmsd_identity_match_cluster_size'] = pd.Series(match_cluster).map(cluster_sizes).to_numpy()
    return out

## Load metadata and outcomes

In [ ]:
low_rmsd_meta = pd.read_csv(LOW_RMSD_TRAIN_PATH, usecols=['pdb_name', 'cluster_ids'])
low_rmsd_train_ids = sorted(low_rmsd_meta['pdb_name'].unique())
id_to_cluster = low_rmsd_meta.set_index('pdb_name')['cluster_ids']
cluster_sizes = low_rmsd_meta['cluster_ids'].value_counts()
print(f'Low-RMSD training samples: {len(low_rmsd_train_ids)}')
print(f'Unique clusters in low-RMSD pool: {cluster_sizes.size}')

rmsds_df = pd.read_csv(RMSD_SUMMARY_PATH)
rmsds_df = rmsds_df[rmsds_df['status'] == 'ok'].copy()
rmsds_df = rmsds_df.rename(columns={'pdb_name': 'test_id'})
print(f'Test samples with status=ok: {len(rmsds_df)}')

test_meta = pd.read_csv(TEST_META_PATH, usecols=['pdb_name', 'CDRH3', 'concat_CDR'])
test_meta = test_meta.rename(columns={'pdb_name': 'test_id'})
test_meta['cdrh3_len'] = test_meta['CDRH3'].str.len()
test_meta['concat_cdr_len'] = test_meta['concat_CDR'].str.len()

train_meta = pd.read_csv(TRAIN_META_PATH, usecols=['pdb_name', 'CDRH3', 'concat_CDR'])
train_meta = train_meta[train_meta['pdb_name'].isin(low_rmsd_train_ids)].copy()
train_meta['cdrh3_len'] = train_meta['CDRH3'].str.len()
train_meta['concat_cdr_len'] = train_meta['concat_CDR'].str.len()
train_meta = train_meta.set_index('pdb_name').loc[low_rmsd_train_ids].reset_index()

features_df = rmsds_df.merge(test_meta[['test_id', 'cdrh3_len', 'concat_cdr_len']], on='test_id', how='inner')
test_ids = features_df['test_id'].tolist()
print(f'Rows after merging test metadata: {len(features_df)}')

In [ ]:
7292/8794

## Load similarity matrices and restrict to low-RMSD train columns

In [ ]:
print('Loading Wasserstein matrix...')
dist_df = pd.read_csv(WASSERSTEIN_MATRIX_PATH, index_col=0)
print('Loading identity matrix...')
id_df = pd.read_csv(IDENTITY_MATRIX_PATH, index_col=0)

missing_train = set(low_rmsd_train_ids) - set(dist_df.columns)
if missing_train:
    raise ValueError(f'{len(missing_train)} low-RMSD train IDs missing from Wasserstein matrix')

missing_test = set(test_ids) - set(dist_df.index)
if missing_test:
    raise ValueError(f'{len(missing_test)} test IDs missing from Wasserstein matrix')

dist_mat = dist_df.loc[test_ids, low_rmsd_train_ids].to_numpy(dtype=np.float64)
id_mat = id_df.loc[test_ids, low_rmsd_train_ids].to_numpy(dtype=np.float64)
print(f'Filtered matrices shape: {dist_mat.shape} (test x low-RMSD train)')

train_cdrh3_len = train_meta['cdrh3_len'].to_numpy()
train_concat_len = train_meta['concat_cdr_len'].to_numpy()
test_cdrh3_len = features_df['cdrh3_len'].to_numpy()
test_concat_len = features_df['concat_cdr_len'].to_numpy()

## Compute restricted similarity features

In [ ]:
wd_features = compute_distance_features(
    dist_mat,
    low_rmsd_train_ids,
    train_cdrh3_len,
    train_concat_len,
    test_cdrh3_len,
    test_concat_len,
    id_to_cluster,
    cluster_sizes,
)
id_features = compute_similarity_features(
    id_mat,
    low_rmsd_train_ids,
    train_cdrh3_len,
    train_concat_len,
    test_cdrh3_len,
    test_concat_len,
    id_to_cluster,
    cluster_sizes,
)

for feat_dict in (wd_features, id_features):
    for col, values in feat_dict.items():
        features_df[col] = values

features_df.head()

In [ ]:
features_df.to_csv(FEATURES_OUT_PATH, index=False)
print(f'Saved {len(features_df)} rows to {FEATURES_OUT_PATH}')

## Sanity checks

In [ ]:
cols=['low_rmsd_wasserstein_mean_knn_3', 'low_rmsd_wasserstein_mean_knn_5', 'low_rmsd_wasserstein_mean_knn_10']
t=features_df[~(features_df['low_rmsd_wasserstein_mean_knn_3'] <= features_df['low_rmsd_wasserstein_mean_knn_5'])]
t[cols]

In [ ]:
full_dist_best = dist_df.loc[test_ids].min(axis=1).to_numpy()
full_id_best = id_df.loc[test_ids].max(axis=1).to_numpy()

assert (features_df['low_rmsd_best_wasserstein_distance'] >= full_dist_best - 1e-6).all()
assert (features_df['low_rmsd_best_identity_concat_cdr'] <= full_id_best + 1e-9).all()
assert (features_df['low_rmsd_wasserstein_match_cluster_size'] >= 1).all()
assert (features_df['low_rmsd_identity_match_cluster_size'] >= 1).all()
assert (features_df['low_rmsd_wasserstein_mean_knn_3'] <= features_df['low_rmsd_wasserstein_mean_knn_5'] + 1e-6).all()
assert (features_df['low_rmsd_wasserstein_mean_knn_5'] <= features_df['low_rmsd_wasserstein_mean_knn_10'] + 1e-6).all()
assert (features_df['low_rmsd_identity_mean_topk_3'] >= features_df['low_rmsd_identity_mean_topk_5'] - 1e-6).all()
assert (features_df['low_rmsd_identity_mean_topk_5'] >= features_df['low_rmsd_identity_mean_topk_10'] - 1e-6).all()
print('Sanity checks passed.')

print('\nWasserstein radius count summary:')
display(features_df[[c for c in features_df.columns if 'wasserstein_lte' in c]].describe().T)
print('\nIdentity threshold count summary:')
display(features_df[[c for c in features_df.columns if 'identity_gt' in c]].describe().T)

## Univariate regression

In [ ]:
usable_wd = [c for c in FEATURE_COLS_WASSERSTEIN if c in features_df.columns and features_df[c].notna().any()]
usable_id = [c for c in FEATURE_COLS_IDENTITY if c in features_df.columns and features_df[c].notna().any()]
usable_cov = [c for c in COVARIATES if c in features_df.columns and features_df[c].notna().any()]

uni_rows = []
for feat in usable_wd:
    stats = univariate_linear_regression(features_df[feat], features_df[TARGET])
    uni_rows.append({'metric_family': 'wasserstein', 'feature': feat, **stats})
for feat in usable_id:
    stats = univariate_linear_regression(features_df[feat], features_df[TARGET])
    uni_rows.append({'metric_family': 'identity', 'feature': feat, **stats})
for feat in usable_cov:
    stats = univariate_linear_regression(features_df[feat], features_df[TARGET])
    uni_rows.append({'metric_family': 'covariate', 'feature': feat, **stats})

uni_results = pd.DataFrame(uni_rows).sort_values('r2', ascending=False)
uni_results[['metric_family', 'feature', 'r2_percent', 'adj_r2', 'slope', 'intercept', 'p_value', 'n']]

In [ ]:
top_n = 12
plot_df = uni_results.head(top_n).copy()
family_colors = {'wasserstein': 'C0', 'identity': 'C1', 'covariate': 'C2'}
colors = plot_df['metric_family'].map(family_colors)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df['feature'][::-1], plot_df['r2_percent'][::-1], color=colors[::-1])
ax.set_xlabel('R² (% variance explained)')
ax.set_title(f'Top {top_n} univariate predictors of {TARGET} (low-RMSD train pool)')
ax.legend(handles=[plt.Line2D([0], [0], color=family_colors[k], lw=8, label=k)
                   for k in ('wasserstein', 'identity', 'covariate')])
plt.tight_layout()
plt.show()

## Multivariate regression

In [ ]:
def run_multivariate(feature_cols, label):
    cols = [c for c in feature_cols if c in features_df.columns]
    model_df = features_df[[TARGET] + cols].dropna()
    X = model_df[cols].to_numpy()
    y = model_df[TARGET].to_numpy()
    result = multivariate_linear_regression(X, y)
    coef_df = pd.DataFrame({'feature': cols, 'coefficient': result['coefficients']})
    summary = {
        'model': label,
        'n': result['n'],
        'p': result['p'],
        'r2_percent': result['r2_percent'],
        'adj_r2': result['adj_r2'],
        'intercept': result['intercept'],
    }
    return summary, coef_df, model_df

multi_summaries = []
coef_tables = {}

for label, cols in [
    ('wasserstein_only', FEATURE_COLS_WASSERSTEIN),
    ('identity_only', FEATURE_COLS_IDENTITY),
    ('wasserstein_with_covariates', FEATURE_COLS_WASSERSTEIN + COVARIATES),
    ('identity_with_covariates', FEATURE_COLS_IDENTITY + COVARIATES),
    ('covariates_only', COVARIATES),
]:
    summary, coef_df, _ = run_multivariate(cols, label)
    multi_summaries.append(summary)
    coef_tables[label] = coef_df

multi_summary_df = pd.DataFrame(multi_summaries)
multi_summary_df

In [ ]:
for label, coef_df in coef_tables.items():
    print(f'\n=== {label} ===')
    display(coef_df.sort_values('coefficient', key=np.abs, ascending=False))

## Diagnostic scatter plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = features_df['low_rmsd_wasserstein_best_same_cdrh3_len']
y = features_df[TARGET]
m, b = np.polyfit(x, y, 1)
wd_r2 = univariate_linear_regression(x, y)['r2']
axes[0].scatter(x, y, alpha=0.25, s=12)
xs = np.linspace(x.min(), x.max(), 100)
axes[0].plot(xs, m * xs + b, color='C0')
axes[0].set_xlabel('Best Train Embedding Distance with same CDRH3 length')
axes[0].set_ylabel(TARGET)
axes[0].set_title(f'PLM EmbeddingDistance vs CDRH3 RMSD ($R^2$ = {wd_r2:.3f})')
#axes[0].set_title(f'PLM EmbeddingDistance vs CDRH3 RMSD (Spearmann = {0.440376:.3f})')

x = features_df['low_rmsd_identity_best_same_cdrh3_len']
y = features_df[TARGET]
m, b = np.polyfit(x, y, 1)
id_r2 = univariate_linear_regression(x, y)['r2']
axes[1].scatter(x, y, alpha=0.25, s=12)
xs = np.linspace(x.min(), x.max(), 100)
axes[1].plot(xs, m * xs + b, color='C1')
axes[1].set_xlabel('Best low-RMSD concat-CDR  sequence identity with same CDRH3 length')
axes[1].set_ylabel(TARGET)
axes[1].set_title(f'Sequence Identity vs CDRH3 RMSD ($R^2$ = {id_r2:.3f})')
#axes[1].set_title(f'Sequence Identity vs CDRH3 RMSD (Spearmann = {-0.414609:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
wd_spermans=features_df[FEATURE_COLS_WASSERSTEIN+['H_cdr3']].corr(method='spearman')
id_spermans=features_df[FEATURE_COLS_IDENTITY+['H_cdr3']].corr(method='spearman')

In [ ]:
wd_spermans.H_cdr3

In [ ]:
id_spermans.H_cdr3

In [ ]:
features_df.cluster_ids

In [ ]:
id_spermans.H_cdr3

In [ ]:
#spearman correlation 
spearman_corr = features_df[['low_rmsd_wasserstein_best_same_cdrh3_len', 'low_rmsd_identity_best_same_cdrh3_len', 'H_cdr3']].corr(method='spearman')
spearman_corr


In [ ]:
import seaborn as sns

#identitiy violin plot
# Define 0.05-width bins from 0.40 to 0.85
bins = np.arange(0.40, 0.90, 0.05)

# Create bin labels
features_df['identity_bin'] = pd.cut(
    features_df['low_rmsd_best_identity_concat_cdr'],
    bins=bins,
    include_lowest=True
)

plt.figure(figsize=(10, 6))

sns.violinplot(
    data=features_df,
    x='identity_bin',
    y='H_cdr3',
    inner='box',      # shows median and IQR
    cut=0             # don't extend beyond the data
)

plt.title("CDRH3 RMSD of Test Set as a function of Training Similarity")
plt.xlabel("Closest CDRH3 Identity in Training Set")
plt.ylabel("CDRH3 RMSD")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#embedding distance violin plot

features_df["wd_bin"] = pd.qcut(features_df["low_rmsd_best_wasserstein_distance"], q=6)

plt.figure(figsize=(10, 6))

sns.violinplot(
    data=features_df,
    x='wd_bin',
    y='H_cdr3',
    inner='box',      # shows median and IQR
    cut=0             # don't extend beyond the data
)

plt.title("CDRH3 RMSD of Test Set as a function of ESM-C Embedding Distance")
plt.xlabel("Closest CDR Embedding in Training Set")
plt.ylabel("CDRH3 RMSD")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Family-specific univariate R²

Each panel lists every feature in one metric family, sorted by univariate R² (% of `H_cdr3` variance explained). The accent bar is the **multivariate** linear model using **all** features in that family (no length/variance covariates).

In [ ]:
import seaborn as sns
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid', context='notebook')

MULTI_MODEL = {
    'wasserstein': 'wasserstein_only',
    'identity': 'identity_only',
}

PALETTES = {
    'wasserstein': {'uni': '#264653', 'multi': '#E9C46A'},
    'identity': {'uni': '#2A9D8F', 'multi': '#E76F51'},
}

TITLES = {
    'wasserstein': 'PLM Wasserstein distance Features',
    'identity': '% Sequence Identity Features',
}


def pretty_feature_name(feature):
    return feature.removeprefix('low_rmsd_').replace('_', ' ')


def plot_family_r2(family):
    colors = PALETTES[family]
    fam_df = (
        uni_results.loc[uni_results['metric_family'] == family]
        .sort_values('r2_percent', ascending=True)
        .copy()
    )
    fam_df['label'] = fam_df['feature'].map(pretty_feature_name)

    multi_model = MULTI_MODEL[family]
    multi_r2 = float(
        multi_summary_df.loc[multi_summary_df['model'] == multi_model, 'r2_percent'].iloc[0]
    )
    multi_label = 'Multivariate (all features)'

    labels = fam_df['label'].tolist() + [multi_label]
    values = fam_df['r2_percent'].tolist() + [multi_r2]
    bar_colors = [colors['uni']] * len(fam_df) + [colors['multi']]

    fig_h = max(4.5, 0.42 * len(labels) + 1.2)
    fig, ax = plt.subplots(figsize=(9, fig_h))

    y_pos = np.arange(len(labels))
    bars = ax.barh(
        y_pos,
        values,
        color=bar_colors,
        edgecolor='white',
        linewidth=0.6,
        height=0.72,
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_xlabel('R² (% variance explained)')
    ax.set_title(f"{TITLES[family]} → CDRH3 RMSD")
    xmax = max(values) * 1.18 if values else 5
    ax.set_xlim(0, max(5, xmax))

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_width() + 0.15,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%',
            va='center',
            ha='left',
            fontsize=9,
        )

    ax.legend(
        handles=[
            Patch(facecolor=colors['uni'], label='Univariate feature'),
            Patch(facecolor=colors['multi'], label=multi_label),
        ],
        loc='lower right',
        frameon=True,
    )
    plt.tight_layout()
    plt.show()


for fam in ('wasserstein', 'identity'):
    plot_family_r2(fam)

## Test-set concat-CDR embedding geometry

Explore **ESM-C concat-CDR** embeddings for the same test cohort as above (`features_df`). Residue tensors are **mean-pooled** to one 960-D vector per antibody for **PCA / t-SNE / UMAP** plots.

**Caveats:** (1) Wasserstein regression in this notebook uses **VH+VL CDR** distances (`vhvl_wasserstein_cdrs`), not these concat-CDR files—geometry here complements but does not duplicate that analysis. (2) 2D projections use mean-pooled vectors; quartile distance summaries below use **full per-residue W2** from Slurm (`get_train_test_wasserstein.py`). (3) 2D plots are exploratory. (4) `cluster_ids` comes from test metadata, not from low-RMSD train match clusters.

Distance summaries compare **bottom vs top `H_cdr3` quartiles** (middle 50% excluded from quartile labels). Export well/ill meta CSVs, run [`wasserstein_well_ill_quartiles.sbatch`](../plm/wasserstein_well_ill_quartiles.sbatch), then run the W2 summary cell.

Cluster-colored PCA/t-SNE/UMAP panels use a **cluster_id legend under the figure**.


In [ ]:
len(embed_df[embed_df['cluster_ids'] == 3])

In [ ]:
import torch
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

EMBED_MANIFEST_PATH = (
    '/opig-shared/users/lina4783/abb4_experiments/plm/esmc_300m_concat_cdr.csv'
)


def l2_normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)


def load_full_embedding(path):
    embedding = torch.load(path, map_location='cpu', weights_only=True)
    if not isinstance(embedding, torch.Tensor) or embedding.ndim != 2:
        raise ValueError(f'Expected 2D tensor, got {type(embedding)} shape {getattr(embedding, "shape", None)}')
    arr = embedding.detach().cpu().float().numpy()
    if arr.shape[0] == 0:
        raise ValueError('Empty residue embedding')
    return arr


def load_mean_embedding(path):
    return load_full_embedding(path).mean(axis=0)


manifest = pd.read_csv(EMBED_MANIFEST_PATH)
manifest = manifest.drop_duplicates(subset='pdb_name', keep='first').set_index('pdb_name', drop=False)

cluster_meta = pd.read_csv(TEST_META_PATH, usecols=['pdb_name', 'cluster_ids'])
cluster_meta = cluster_meta.rename(columns={'pdb_name': 'test_id'})

embed_df = features_df[['test_id', TARGET]].merge(cluster_meta, on='test_id', how='left')
#embed_df=embed_df[embed_df['cluster_ids'] != 3]

vectors = []
embedding_paths = []
keep_rows = []
skipped = []
for row in embed_df.itertuples(index=False):
    test_id = row.test_id
    if test_id not in manifest.index:
        skipped.append((test_id, 'not in manifest'))
        continue
    path = Path(manifest.loc[test_id, 'embedding_path'])
    if not path.exists():
        skipped.append((test_id, 'missing file'))
        continue
    try:
        vectors.append(load_mean_embedding(path))
        embedding_paths.append(str(path))
        keep_rows.append(row)
    except Exception as exc:
        skipped.append((test_id, str(exc)))

embed_df = pd.DataFrame(keep_rows).reset_index(drop=True)
embed_df['embedding_path'] = embedding_paths
X = np.stack(vectors, axis=0)
X_norm = l2_normalize(X)

q25, q75 = embed_df[TARGET].quantile([0.25, 0.75])
embed_df['pred_group'] = np.select(
    [embed_df[TARGET] <= q25, embed_df[TARGET] >= q75],
    ['well', 'ill'],
    default='middle',
)

print(f'Loaded {len(embed_df)} / {len(features_df)} test embeddings (dim={X.shape[1]})')
print(f'Skipped: {len(skipped)}')
if skipped[:3]:
    print('Skip examples:', skipped[:3])
print(f'Quartile thresholds on {TARGET}: Q25={q25:.3f}, Q75={q75:.3f}')
print(embed_df['pred_group'].value_counts().to_string())


In [ ]:
from pathlib import Path
PLM_DIR = Path('/opig-shared/users/lina4783/abb4_experiments/plm')
WELL_META_PATH = PLM_DIR / 'test_meta_well_predicted.csv'
ILL_META_PATH = PLM_DIR / 'test_meta_ill_predicted.csv'

seq_meta = pd.read_csv(TEST_META_PATH, usecols=['pdb_name', 'concat_CDR'])
quartile_meta = embed_df[['test_id', TARGET, 'pred_group']].merge(
    seq_meta, left_on='test_id', right_on='pdb_name', how='inner'
)

well_meta = quartile_meta.loc[quartile_meta['pred_group'] == 'well', ['pdb_name', 'concat_CDR']]
ill_meta = quartile_meta.loc[quartile_meta['pred_group'] == 'ill', ['pdb_name', 'concat_CDR']]

well_meta.to_csv(WELL_META_PATH, index=False)
ill_meta.to_csv(ILL_META_PATH, index=False)

print(f'Wrote {len(well_meta)} rows -> {WELL_META_PATH}')
print(f'Wrote {len(ill_meta)} rows -> {ILL_META_PATH}')


### Slurm: full-embedding Wasserstein for well / ill quartiles

After exporting meta CSVs above, submit:

`sbatch /opig-shared/users/lina4783/abb4_experiments/plm/wasserstein_well_ill_quartiles.sbatch`

Outputs (each includes `similarity_matrix.csv`):

| Comparison | Output directory |
|------------|------------------|
| within well | `plm/wasserstein_out/well_well_concat_cdr_300m/` |
| within ill | `plm/wasserstein_out/ill_ill_concat_cdr_300m/` |
| well vs ill | `plm/wasserstein_out/well_train_ill_test_concat_cdr_300m/` |


In [ ]:
import umap

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca2 = PCA(n_components=2, random_state=0)
pca_coords = pca2.fit_transform(X_scaled)
pca_var = pca2.explained_variance_ratio_

n_pca_pre = min(50, X_scaled.shape[0], X_scaled.shape[1])
pca_pre = PCA(n_components=n_pca_pre, random_state=0)
X_pca_pre = pca_pre.fit_transform(X_scaled)

perplexity = min(30, max(5, (len(embed_df) - 1) // 3))
tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=0,
)
tsne_coords = tsne.fit_transform(X_pca_pre)

umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='euclidean',
    random_state=0,
)
umap_coords = umap_model.fit_transform(X_pca_pre)

embed_df['pca1'] = pca_coords[:, 0]
embed_df['pca2'] = pca_coords[:, 1]
embed_df['tsne1'] = tsne_coords[:, 0]
embed_df['tsne2'] = tsne_coords[:, 1]
embed_df['umap1'] = umap_coords[:, 0]
embed_df['umap2'] = umap_coords[:, 1]

cluster_sizes = embed_df['cluster_ids'].value_counts()
small_clusters = set(cluster_sizes[cluster_sizes < 5].index)
embed_df['cluster_plot'] = embed_df['cluster_ids'].astype(str)
embed_df.loc[embed_df['cluster_ids'].isin(small_clusters), 'cluster_plot'] = 'small cluster (<5)'

# CLUSTER_LEGEND_BELOW_V2 — legend under figure, not on scatter axes
pca_title_suffix = f'PC1 {pca_var[0]:.1%} + PC2 {pca_var[1]:.1%} variance'

from matplotlib.patches import Patch

cluster_labels = list(embed_df.groupby('cluster_plot', sort=False).groups.keys())
n_clusters_leg = len(cluster_labels)
_cmap = plt.colormaps['tab20']
cluster_colors = {lab: _cmap(i % 20) for i, lab in enumerate(cluster_labels)}

legend_ncol = min(12, max(6, int(np.ceil(np.sqrt(n_clusters_leg)))))
legend_nrows = int(np.ceil(n_clusters_leg / legend_ncol))
fig_height = 14.0 + 0.32 * legend_nrows
bottom_margin = 0.05 + 0.022 * legend_nrows

fig, axes = plt.subplots(3, 2, figsize=(14, fig_height))
print(
    f'PCA/t-SNE/UMAP: {n_clusters_leg} legend entries below panels ({legend_nrows} row(s))'
)

projection_rows = [
    ('pca1', 'pca2', f'PCA ({pca_title_suffix})'),
    ('tsne1', 'tsne2', f't-SNE (perplexity={perplexity})'),
    ('umap1', 'umap2', 'UMAP (on PCA pre-space)'),
]

for row_idx, (xcol, ycol, title_prefix) in enumerate(projection_rows):
    ax_c = axes[row_idx, 0]
    for label, sub in embed_df.groupby('cluster_plot', sort=False):
        ax_c.scatter(sub[xcol], sub[ycol], s=14, alpha=0.55, color=cluster_colors[label])
    ax_c.set_xlabel(xcol)
    ax_c.set_ylabel(ycol)
    ax_c.set_title(f'{title_prefix} — cluster_ids')

    ax_t = axes[row_idx, 1]
    sc = ax_t.scatter(embed_df[xcol], embed_df[ycol], c=embed_df[TARGET], s=14, alpha=0.65, cmap='plasma')
    ax_t.set_xlabel(xcol)
    ax_t.set_ylabel(ycol)
    ax_t.set_title(f'{title_prefix} — {TARGET} (Å)')
    plt.colorbar(sc, ax=ax_t, label=TARGET)

legend_handles = [
    Patch(facecolor=cluster_colors[lab], edgecolor='0.4', linewidth=0.4, label=lab)
    for lab in cluster_labels
]
fig.subplots_adjust(hspace=0.42, wspace=0.28, top=0.96, bottom=bottom_margin)
fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.0),
    ncol=legend_ncol,
    fontsize=6,
    title='cluster_ids (point color)',
    title_fontsize=9,
    frameon=True,
    fancybox=False,
    edgecolor='0.75',
)

plt.show()


### t-SNE hyperparameter sweep

Six t-SNE fits on the same preprocessing as the main figure (**StandardScaler** → **PCA** (≤50 components) → t-SNE on PCA space). Each row shows **cluster_ids** (left) and **`H_cdr3`** (right).

Expect **~10–20 minutes** for all six configs on ~2.2k test antibodies. Requires the embedding load cell (`embed_df`, `X`) to have been run first; does not require the full PCA/t-SNE/UMAP figure cell.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Patch

n = len(embed_df)
p_default = min(30, max(5, (n - 1) // 3))

TSNE_DEFAULTS = dict(
    n_components=2,
    init='pca',
    learning_rate='auto',
    early_exaggeration=12.0,
    random_state=0,
    max_iter=1000,
)

TSNE_CONFIGS = [
    {'label': f'baseline (perplexity={p_default})', 'perplexity': p_default},
    {'label': 'perplexity=5', 'perplexity': 5},
    {'label': 'perplexity=15', 'perplexity': 15},
    {'label': 'perplexity=50', 'perplexity': min(50, n - 1)},
    {'label': 'perplexity=15, init=random', 'perplexity': 15, 'init': 'random'},
    {
        'label': 'perplexity=30, exaggeration=30, lr=200',
        'perplexity': min(30, n - 1),
        'early_exaggeration': 30.0,
        'learning_rate': 200,
    },
]

if 'cluster_plot' not in embed_df.columns:
    cluster_sizes = embed_df['cluster_ids'].value_counts()
    small_clusters = set(cluster_sizes[cluster_sizes < 5].index)
    embed_df['cluster_plot'] = embed_df['cluster_ids'].astype(str)
    embed_df.loc[embed_df['cluster_ids'].isin(small_clusters), 'cluster_plot'] = 'small cluster (<5)'

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
n_pca_pre = min(50, X_scaled.shape[0], X_scaled.shape[1])
pca_pre = PCA(n_components=n_pca_pre, random_state=0)
X_pca_pre = pca_pre.fit_transform(X_scaled)

cluster_labels = list(embed_df.groupby('cluster_plot', sort=False).groups.keys())
n_clusters_leg = len(cluster_labels)
_cmap = plt.colormaps['tab20']
cluster_colors = {lab: _cmap(i % 20) for i, lab in enumerate(cluster_labels)}

legend_ncol = min(12, max(6, int(np.ceil(np.sqrt(n_clusters_leg)))))
legend_nrows = int(np.ceil(n_clusters_leg / legend_ncol))
fig_height = 3.2 * len(TSNE_CONFIGS) + 0.35 * legend_nrows
bottom_margin = 0.03 + 0.018 * legend_nrows

fig, axes = plt.subplots(len(TSNE_CONFIGS), 2, figsize=(14, fig_height))
if len(TSNE_CONFIGS) == 1:
    axes = np.array([axes])

tsne_sweep_results = []

for i, cfg in enumerate(TSNE_CONFIGS):
    label = cfg['label']
    params = {**TSNE_DEFAULTS, **{k: v for k, v in cfg.items() if k != 'label'}}
    print(f'[{i + 1}/{len(TSNE_CONFIGS)}] Fitting t-SNE: {label} ...', flush=True)
    tsne = TSNE(**params)
    coords = tsne.fit_transform(X_pca_pre)
    tsne_sweep_results.append({'label': label, 'coords': coords, 'params': params})

    ax_c, ax_t = axes[i, 0], axes[i, 1]
    for clab, sub in embed_df.groupby('cluster_plot', sort=False):
        ax_c.scatter(coords[sub.index, 0], coords[sub.index, 1], s=12, alpha=0.55, color=cluster_colors[clab])
    ax_c.set_xlabel('t-SNE 1')
    ax_c.set_ylabel('t-SNE 2')
    ax_c.set_title(f't-SNE — {label} — cluster_ids')

    sc = ax_t.scatter(coords[:, 0], coords[:, 1], c=embed_df[TARGET], s=12, alpha=0.65, cmap='plasma')
    ax_t.set_xlabel('t-SNE 1')
    ax_t.set_ylabel('t-SNE 2')
    ax_t.set_title(f't-SNE — {label} — {TARGET} (Å)')
    plt.colorbar(sc, ax=ax_t, label=TARGET)

legend_handles = [
    Patch(facecolor=cluster_colors[lab], edgecolor='0.4', linewidth=0.4, label=lab)
    for lab in cluster_labels
]
fig.subplots_adjust(hspace=0.45, wspace=0.28, top=0.98, bottom=bottom_margin)
fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.0),
    ncol=legend_ncol,
    fontsize=6,
    title='cluster_ids (point color)',
    title_fontsize=9,
    frameon=True,
    fancybox=False,
    edgecolor='0.75',
)
plt.show()
print('Done. Results in tsne_sweep_results (list of dicts with label, coords, params).')


### t-SNE (config 6) with emphasis on low RMSD colors

Same t-SNE as sweep row **perplexity=30, exaggeration=30, lr=200**. **H_cdr3** uses `PowerNorm` (no hard cutoff): full RMSD range stays on the colorbar, but **lower values get more color separation** and **high values change more slowly** than with linear scaling.

Reference interpretation (not color boundaries): **≤1.5 Å** very good, **≤2.5 Å** good; values around **4.5 Å** and above are increasingly poor but remain distinguishable at the top of the scale.

Run the hyperparameter sweep cell first to reuse coordinates without refitting.


In [ ]:
from matplotlib.colors import PowerNorm
from matplotlib.patches import Patch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

TSNE_FOCUS_LABEL = 'perplexity=30, exaggeration=30, lr=200'
RMSD_COLOR_GAMMA = 0.5  # lower = more contrast at low RMSD; try 0.4–0.6

if 'cluster_plot' not in embed_df.columns:
    cluster_sizes = embed_df['cluster_ids'].value_counts()
    small_clusters = set(cluster_sizes[cluster_sizes < 5].index)
    embed_df['cluster_plot'] = embed_df['cluster_ids'].astype(str)
    embed_df.loc[embed_df['cluster_ids'].isin(small_clusters), 'cluster_plot'] = 'small cluster (<5)'

coords = None
if 'tsne_sweep_results' in dir() and tsne_sweep_results:
    for entry in tsne_sweep_results:
        if entry.get('label') == TSNE_FOCUS_LABEL:
            coords = entry['coords']
            print(f'Reusing t-SNE coords from sweep: {TSNE_FOCUS_LABEL}')
            break
    if coords is None:
        coords = tsne_sweep_results[-1]['coords']
        print(f'Label not found; using last sweep result: {tsne_sweep_results[-1]["label"]!r}')

if coords is None:
    print(f'Refitting t-SNE: {TSNE_FOCUS_LABEL} ...', flush=True)
    n = len(embed_df)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    n_pca_pre = min(50, X_scaled.shape[0], X_scaled.shape[1])
    X_pca_pre = PCA(n_components=n_pca_pre, random_state=0).fit_transform(X_scaled)
    tsne = TSNE(
        n_components=2,
        perplexity=min(30, n - 1),
        init='pca',
        learning_rate=200,
        early_exaggeration=30.0,
        random_state=0,
        max_iter=1000,
    )
    coords = tsne.fit_transform(X_pca_pre)

rmsd_vmin = 0.0
rmsd_vmax = float(embed_df[TARGET].max())
rmsd_norm = PowerNorm(gamma=RMSD_COLOR_GAMMA, vmin=rmsd_vmin, vmax=rmsd_vmax)

cluster_labels = list(embed_df.groupby('cluster_plot', sort=False).groups.keys())
n_clusters_leg = len(cluster_labels)
_cmap = plt.colormaps['tab20']
cluster_colors = {lab: _cmap(i % 20) for i, lab in enumerate(cluster_labels)}

legend_ncol = min(12, max(6, int(np.ceil(np.sqrt(n_clusters_leg)))))
legend_nrows = int(np.ceil(n_clusters_leg / legend_ncol))
bottom_margin = 0.12 + 0.022 * legend_nrows

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax_c, ax_r = axes[0], axes[1]

for clab, sub in embed_df.groupby('cluster_plot', sort=False):
    ax_c.scatter(coords[sub.index, 0], coords[sub.index, 1], s=16, alpha=0.55, color=cluster_colors[clab])
ax_c.set_xlabel('t-SNE 1')
ax_c.set_ylabel('t-SNE 2')
ax_c.set_title(f't-SNE of ESM-C Embeddings by Cluster')

sc = ax_r.scatter(
    coords[:, 0], coords[:, 1],
    c=embed_df[TARGET], s=16, alpha=0.65, cmap='plasma', norm=rmsd_norm,
)
ax_r.set_xlabel('t-SNE 1')
ax_r.set_ylabel('t-SNE 2')
ax_r.set_title(f't-SNE of ESM-C Embeddings by CDRH3 RMSD')
cb = plt.colorbar(sc, ax=ax_r, label=f'CDRH3 RMSD (Å)')
for ref in (1.5, 2.5, 4.5):
    if rmsd_vmin <= ref <= rmsd_vmax:
        cb.ax.axhline(ref, color='white', linewidth=0.8, alpha=0.7)

legend_handles = [
    Patch(facecolor=cluster_colors[lab], edgecolor='0.4', linewidth=0.4, label=lab)
    for lab in cluster_labels
]
fig.subplots_adjust(wspace=0.28, bottom=bottom_margin)
fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.0),
    ncol=legend_ncol,
    fontsize=6,
    title='cluster_ids (point color)',
    title_fontsize=9,
    frameon=True,
    fancybox=False,
    edgecolor='0.75',
)
plt.show()
print(f'RMSD color scale: [{rmsd_vmin:.2f}, {rmsd_vmax:.2f}] Å with PowerNorm(gamma={RMSD_COLOR_GAMMA})')


In [ ]:
from pathlib import Path

W2_OUT = Path('/opig-shared/users/lina4783/abb4_experiments/plm/wasserstein_out')
W2_WELL_WELL = W2_OUT / 'well_well_concat_cdr_300m' / 'similarity_matrix.csv'
W2_ILL_ILL = W2_OUT / 'ill_ill_concat_cdr_300m' / 'similarity_matrix.csv'
W2_WELL_ILL = W2_OUT / 'well_train_ill_test_concat_cdr_300m' / 'similarity_matrix.csv'

required = {
    'within well': W2_WELL_WELL,
    'within ill': W2_ILL_ILL,
    'well vs ill': W2_WELL_ILL,
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing Wasserstein matrices for: ' + ', '.join(missing) +
        '. Export meta CSVs and run wasserstein_well_ill_quartiles.sbatch first.'
    )


def upper_triangle_values(sim_df):
    ids = sim_df.index.tolist()
    mat = sim_df.loc[ids, ids].to_numpy(dtype=float)
    i, j = np.triu_indices(len(ids), k=1)
    return mat[i, j]


def flatten_train_test(sim_df):
    return sim_df.to_numpy(dtype=float).ravel()


well_well = pd.read_csv(W2_WELL_WELL, index_col=0)
ill_ill = pd.read_csv(W2_ILL_ILL, index_col=0)
well_ill = pd.read_csv(W2_WELL_ILL, index_col=0)

dist_rows = []
for label, vals in [
    ('within_well (bottom quartile)', upper_triangle_values(well_well)),
    ('within_ill (top quartile)', upper_triangle_values(ill_ill)),
    ('between_well_ill', flatten_train_test(well_ill)),
]:
    dist_rows.append({
        'pair_type': label,
        'mean_wasserstein_w2': float(np.mean(vals)),
        'median_wasserstein_w2': float(np.median(vals)),
        'n_pairs': int(len(vals)),
    })

dist_summary_df = pd.DataFrame(dist_rows)
display(dist_summary_df)

fig, ax = plt.subplots(figsize=(7, 4))
plot_parts = [
    ('within_well', upper_triangle_values(well_well)),
    ('within_ill', upper_triangle_values(ill_ill)),
    ('between', flatten_train_test(well_ill)),
]
ax.violinplot([p[1] for p in plot_parts], showmeans=True, showmedians=True)
ax.set_xticks(np.arange(1, 4))
ax.set_xticklabels([p[0] for p in plot_parts])
ax.set_ylabel('Wasserstein-2 (full concat-CDR residues, uniform OT, sq Euclidean cost)')
ax.set_title('Pairwise W2: well vs ill quartiles (middle 50% excluded from groups)')
plt.tight_layout()
plt.show()


In [ ]:
import math
from scipy.spatial.distance import cdist
import ot


def wasserstein2_uniform(embed_a, embed_b):
    n_a, n_b = embed_a.shape[0], embed_b.shape[0]
    if n_a == 0 or n_b == 0:
        return float('nan')
    weights_a = np.full(n_a, 1.0 / n_a, dtype=np.float64)
    weights_b = np.full(n_b, 1.0 / n_b, dtype=np.float64)
    cost = cdist(embed_a, embed_b, metric='sqeuclidean')
    w2_squared = ot.emd2(weights_a, weights_b, cost)
    return float(math.sqrt(max(w2_squared, 0.0)))


def nearest_neighbor_w2(query_idx, embed_df):
    query_path = embed_df.loc[query_idx, 'embedding_path']
    query_emb = load_full_embedding(query_path)
    best_j = None
    best_d = math.inf
    for j, row in embed_df.iterrows():
        if j == query_idx:
            continue
        d = wasserstein2_uniform(query_emb, load_full_embedding(row['embedding_path']))
        if d < best_d:
            best_d = d
            best_j = j
    return best_j, best_d


for group_label, ascending in [
    ('best predicted (lowest H_cdr3)', True),
    ('worst predicted (highest H_cdr3)', False),
]:
    idx = embed_df[TARGET].sort_values(ascending=ascending).index[0]
    nn_idx, nn_dist = nearest_neighbor_w2(idx, embed_df)
    print(
        f"{group_label}: {embed_df.loc[idx, 'test_id']} ({TARGET}={embed_df.loc[idx, TARGET]:.3f}) "
        f"→ nearest neighbor {embed_df.loc[nn_idx, 'test_id']} "
        f"(W2={nn_dist:.4f})"
    )
